# Mashable Article Popularity Prediction

**Competition task:** predict whether an article will get **≥ 1400** shares (binary classification).  
**Metric:** ROC-AUC.  
**Important:** the test set is newer than the train set → strong **temporal shift**.

| | |
|---|---|
| Train size | 31,715 |
| Test size | 7,929 |
| Positive rate (train) | ≈ 54.7% |
| Final result | **1st place** (best of 4 public submissions) |

This notebook is the full experimental log: each section states a **hypothesis**, runs **code**, shows **tables/plots**, and ends with a **takeaway**.  
Dead-end experiments are kept (sometimes commented) so the decision path is reproducible.


## 0. Setup: imports and helpers


In [ ]:
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import TimeSeriesSplit, cross_val_predict, StratifiedKFold
from scipy.stats import rankdata

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12

def find(name: str) -> str:
    """Locate a competition file under /kaggle/input or the current directory."""
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        hits = glob.glob(f"./**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found — add competition data")
    return hits[0]

def auc(y_true, y_prob) -> float:
    return float(roc_auc_score(y_true, y_prob))


## 1. Load data and first look

**Hypothesis:** publish dates matter; sorting train chronologically is required for honest validation.


In [ ]:
train = pd.read_csv(find("train.csv"))
test  = pd.read_csv(find("test.csv"))

train["publish_date"] = pd.to_datetime(train["publish_date"])
test["publish_date"]  = pd.to_datetime(test["publish_date"])
train = train.sort_values("publish_date").reset_index(drop=True)

print("train:", train.shape, " | test:", test.shape)
print("positive rate:", round(train["target"].mean(), 4))
print("date range train:", train["publish_date"].min().date(), "→", train["publish_date"].max().date())
print("date range test: ", test["publish_date"].min().date(),  "→", test["publish_date"].max().date())
train.head(3)


In [ ]:
# Target balance and channel distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

train["target"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"], rot=0
)
axes[0].set_title("Target balance (0 = not popular, 1 = popular)")
axes[0].set_xlabel("target")
axes[0].set_ylabel("count")

ch = train.groupby("channel")["target"].agg(["mean", "count"]).sort_values("count", ascending=False)
ch["mean"].plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Positive rate by channel")
axes[1].set_xlabel("P(target=1)")
plt.tight_layout()
plt.show()

print(ch.round(3).to_string())


In [ ]:
# Articles over time
fig, ax = plt.subplots(figsize=(11, 3.5))
train.set_index("publish_date").resample("W")["target"].agg(["count", "mean"]).plot(
    ax=ax, secondary_y="mean"
)
ax.set_title("Weekly volume (left) and positive rate (right)")
ax.set_ylabel("article count")
ax.right_ax.set_ylabel("positive rate")
plt.tight_layout()
plt.show()


**Takeaway:** classes are almost balanced. Channel and time both affect the positive rate → calendar and channel features are candidates. Test starts after train → must validate with a **time-aware** split.


## 2. Feature engineering

**Hypothesis:**
- token ratios must be in [0, 1] (raw data contains outliers up to 1000);
- `-1` in keyword min fields means “no data”;
- heavy-tailed counts benefit from `log1p`;
- density features (images/links per 100 words) and weekday/channel one-hots add signal.


In [ ]:
LOG_COLS = [
    "n_tokens_content", "num_hrefs", "num_self_hrefs", "num_imgs", "num_videos",
    "kw_max_min", "kw_avg_min", "kw_min_max", "kw_max_max", "kw_avg_max",
    "kw_min_avg", "kw_max_avg", "kw_avg_avg",
    "self_reference_min_shares", "self_reference_max_shares", "self_reference_avg_shares",
]

def prepare(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    for c in ["n_unique_tokens", "n_non_stop_words", "n_non_stop_unique_tokens"]:
        d[c] = d[c].clip(0, 1)

    for c in ["kw_min_min", "kw_avg_min"]:
        d[c] = d[c].where(d[c] >= 0)  # -1 → NaN

    words = d["n_tokens_content"].clip(lower=1)
    d["imgs_per_100w"]  = d["num_imgs"]  / words * 100
    d["links_per_100w"] = d["num_hrefs"] / words * 100
    d["self_link_share"] = d["num_self_hrefs"] / d["num_hrefs"].clip(lower=1)
    d["kw_best_vs_avg"] = np.log1p(d["kw_max_avg"]) - np.log1p(d["kw_avg_avg"])

    for c in LOG_COLS:
        d[c] = np.log1p(d[c].clip(lower=0))

    d["month"] = d["publish_date"].dt.month
    d["is_weekend"] = d["weekday"].isin(["Saturday", "Sunday"]).astype(int)
    d = pd.get_dummies(d, columns=["weekday", "channel"], dtype=int)
    return d.drop(columns=["publish_date", "id"])

y = train["target"].values
X = prepare(train.drop(columns=["target"]))
X_test = prepare(test).reindex(columns=X.columns, fill_value=0)
print("X shape:", X.shape, "| X_test shape:", X_test.shape)
print("feature list sample:", list(X.columns[:12]), "...")


**Takeaway:** ~60+ features after encoding. Next compress correlated keyword/sentiment blocks.


## 3. PCA on keyword and sentiment groups

**Hypothesis:** 8 keyword stats and 12 polarity stats are highly correlated; a few principal components keep most variance and reduce noise for trees.


In [ ]:
KW_COLS = [
    "kw_max_min", "kw_avg_min", "kw_min_max", "kw_max_max",
    "kw_avg_max", "kw_min_avg", "kw_max_avg", "kw_avg_avg",
]
SENT_COLS = [
    "global_subjectivity", "global_sentiment_polarity",
    "global_rate_positive_words", "global_rate_negative_words",
    "rate_positive_words", "rate_negative_words",
    "avg_positive_polarity", "min_positive_polarity", "max_positive_polarity",
    "avg_negative_polarity", "min_negative_polarity", "max_negative_polarity",
]

pca_info = {}
for name, cols, n in [("kw", KW_COLS, 3), ("sent", SENT_COLS, 4)]:
    scaler = StandardScaler().fit(X[cols].fillna(0))
    pca = PCA(n_components=n).fit(scaler.transform(X[cols].fillna(0)))
    pca_info[name] = pca.explained_variance_ratio_
    print(f"{name}: explained variance = {pca.explained_variance_ratio_.sum():.3f}",
          list(np.round(pca.explained_variance_ratio_, 3)))
    for df in (X, X_test):
        z = pca.transform(scaler.transform(df[cols].fillna(0)))
        for i in range(n):
            df[f"{name}_pc{i}"] = z[:, i]

# Plot cumulative explained variance
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (name, ratios) in zip(axes, pca_info.items()):
    cum = np.cumsum(ratios)
    ax.bar(range(1, len(ratios) + 1), ratios, color="#4C72B0", label="per component")
    ax.plot(range(1, len(cum) + 1), cum, "o-", color="#DD8452", label="cumulative")
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("component")
    ax.set_title(f"PCA — {name} block")
    ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
print("total features after PCA:", X.shape[1])


**Takeaway:** keyword block ≈ 91% variance in 3 PCs, sentiment ≈ 80% in 4 PCs. Original columns are kept as well (trees handle redundancy).


## 4. Baseline LightGBM — single chronological 80/20 split

**Hypothesis:** a conservative LightGBM (few leaves, regularization, column subsample) is a stable starting point.  
Validation uses the **last 20% of train by time** (closest proxy to the real test).


In [ ]:
split = int(len(X) * 0.8)
X_tr, X_va = X.iloc[:split], X.iloc[split:]
y_tr, y_va = y[:split], y[split:]
print(f"train rows: {len(X_tr)} | val rows: {len(X_va)}")

params = dict(
    objective="binary",
    learning_rate=0.03,
    n_estimators=2000,
    num_leaves=15,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.5,
    reg_lambda=5,
    random_state=42,
    verbose=-1,
)

model = lgb.LGBMClassifier(**params)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
p_lgb_va = model.predict_proba(X_va)[:, 1]
base_auc = auc(y_va, p_lgb_va)
print(f"LightGBM validation AUC: {base_auc:.4f}")
print(f"best iteration: {model.best_iteration_}")


In [ ]:
# ROC curve + top feature importances
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

fpr, tpr, _ = roc_curve(y_va, p_lgb_va)
axes[0].plot(fpr, tpr, color="#4C72B0", lw=2, label=f"LGBM AUC={base_auc:.4f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].set_title("ROC — validation (last 20% by time)")
axes[0].legend()

imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True).tail(15)
imp.plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Top-15 feature importances (LightGBM)")
plt.tight_layout()
plt.show()


**Takeaway:** baseline ≈ **0.7518**. Strong features: keyword averages, self-reference shares, LDA topics, month.


## 5. CatBoost and rank blend

**Hypothesis:** CatBoost has a different bias; mixing **ranks** (not raw probabilities) of the two models improves ranking quality for AUC.


In [ ]:
cb = CatBoostClassifier(
    iterations=2000, learning_rate=0.03, depth=6,
    eval_metric="AUC", verbose=0, random_seed=42,
)
cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=100)
p_cb_va = cb.predict_proba(X_va)[:, 1]
cb_auc = auc(y_va, p_cb_va)
print(f"CatBoost validation AUC: {cb_auc:.4f}")
print(f"best iteration: {cb.get_best_iteration() + 1}")

blend_rows = []
for w in np.linspace(0, 1, 11):
    blend = w * rankdata(p_lgb_va) + (1 - w) * rankdata(p_cb_va)
    blend_rows.append({"LGBM_weight": round(w, 1), "AUC": auc(y_va, blend)})

blend_df = pd.DataFrame(blend_rows)
print(blend_df.to_string(index=False))
best_row = blend_df.loc[blend_df["AUC"].idxmax()]
print(f"\nbest blend: weight={best_row['LGBM_weight']}  AUC={best_row['AUC']:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

blend_df.plot(x="LGBM_weight", y="AUC", marker="o", ax=axes[0], color="#C44E52", legend=False)
axes[0].axhline(base_auc, ls="--", color="#4C72B0", label=f"LGBM {base_auc:.4f}")
axes[0].axhline(cb_auc,   ls="--", color="#55A868", label=f"CatBoost {cb_auc:.4f}")
axes[0].set_title("Rank-blend AUC vs LGBM weight")
axes[0].set_ylabel("validation AUC")
axes[0].legend()

axes[1].scatter(p_lgb_va, p_cb_va, s=4, alpha=0.25, c="#4C72B0")
axes[1].set_xlabel("LightGBM probability")
axes[1].set_ylabel("CatBoost probability")
axes[1].set_title("Model agreement on validation")
plt.tight_layout()
plt.show()


**Takeaway:** CatBoost alone ≈ **0.7542**; best rank blend around weight **0.3** → ≈ **0.7545**.  
Difference between weights 0.3–0.5 is tiny (~0.0003), so either is fine for submission.


## 6. Adversarial validation — measuring train/test shift

**Hypothesis:** if a classifier can tell train from test almost perfectly, covariate shift is severe. Top features of that classifier are the ones that “leak” time.


In [ ]:
adv_X = pd.concat([X, X_test], ignore_index=True)
adv_y = np.r_[np.zeros(len(X)), np.ones(len(X_test))]

adv = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15, verbose=-1)
adv_p = cross_val_predict(
    adv, adv_X, adv_y,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    method="predict_proba",
)[:, 1]
adv_auc = auc(adv_y, adv_p)
print(f"Adversarial AUC (train vs test): {adv_auc:.4f}")

adv.fit(adv_X, adv_y)
adv_imp = pd.Series(adv.feature_importances_, index=adv_X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
adv_imp.head(15).sort_values().plot(kind="barh", ax=ax, color="#C44E52")
ax.set_title(f"Top features that separate train from test (adv AUC={adv_auc:.3f})")
plt.tight_layout()
plt.show()
print(adv_imp.head(15).to_string())


**Takeaway:** adversarial AUC ≈ **0.998** → almost perfect separation.  
Top drivers: `month`, keyword min/avg fields.  
**Next question:** does *removing* them help the real target, or only the artificial “train vs test” task?


## 7. Experiment: drop time-leaking features?

**Hypothesis (single 80/20):** removing `month`, `kw_min_min`, `kw_avg_min` should raise validation AUC if those features only overfit the train period.

We compare four feature sets on the **same** chronological 80/20 split.


In [ ]:
def val_auc_on_cols(cols, label=""):
    m = lgb.LGBMClassifier(**{**params, "n_estimators": 2000})
    m.fit(
        X_tr[cols], y_tr,
        eval_set=[(X_va[cols], y_va)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    score = auc(y_va, m.predict_proba(X_va[cols])[:, 1])
    return score, m.best_iteration_

all_cols = list(X.columns)
kw_all = [c for c in all_cols if c.startswith("kw_")]
DROP = ["month", "kw_min_min", "kw_avg_min"]
FEATS = [c for c in all_cols if c not in DROP]

drop_sets = {
    "all features":              all_cols,
    "no month":                  [c for c in all_cols if c != "month"],
    "no month + kw_min/avg_min": FEATS,
    "no month + all kw_*":       [c for c in all_cols if c != "month" and c not in kw_all],
}

single_split_rows = []
for name, cols in drop_sets.items():
    score, ntree = val_auc_on_cols(cols)
    single_split_rows.append({"feature_set": name, "n_features": len(cols), "AUC": score, "trees": ntree})
    print(f"{name:30s}  AUC={score:.4f}  trees={ntree}  feats={len(cols)}")

single_split_df = pd.DataFrame(single_split_rows)
single_split_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]
ax.barh(single_split_df["feature_set"], single_split_df["AUC"], color=colors)
ax.set_xlim(0.73, 0.76)
ax.set_xlabel("validation AUC (single 80/20)")
ax.set_title("Feature-set comparison — one chronological split")
for i, row in single_split_df.iterrows():
    ax.text(row["AUC"] + 0.0003, i, f'{row["AUC"]:.4f}', va="center", fontsize=9)
plt.tight_layout()
plt.show()


**On one split:** dropping `month` + two kw fields looks best (≈ **0.7549**).  
Dropping *all* keyword features hurts badly (≈ 0.737).  

**But** a single split can lie. We re-check with **multiple time blocks** next.


## 8. Honest check: TimeSeriesSplit (5 folds)

**Hypothesis:** if the gain from dropping features is real, it will show up as a higher **mean** AUC across several expanding time windows — not only on the last 20%.


In [ ]:
def time_block_scores(cols, n_splits=5):
    """Return list of fold AUCs for LightGBM on given columns."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), 1):
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X.iloc[tr_idx][cols], y[tr_idx],
            eval_set=[(X.iloc[va_idx][cols], y[va_idx])],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )
        s = auc(y[va_idx], m.predict_proba(X.iloc[va_idx][cols])[:, 1])
        scores.append(s)
        print(f"  fold {fold}: AUC={s:.4f}")
    print(f"  mean={np.mean(scores):.4f} ± {np.std(scores):.4f}")
    return scores

print("=== all features ===")
scores_all = time_block_scores(all_cols)

print("\n=== FEATS (no month / kw_min_min / kw_avg_min) ===")
scores_drop = time_block_scores(FEATS)


In [ ]:
cv_df = pd.DataFrame({
    "fold": list(range(1, 6)) * 2,
    "AUC": scores_all + scores_drop,
    "feature_set": ["all features"] * 5 + ["dropped temporal"] * 5,
})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(data=cv_df, x="fold", y="AUC", hue="feature_set", ax=axes[0], palette=["#4C72B0", "#DD8452"])
axes[0].set_ylim(0.65, 0.80)
axes[0].set_title("Per-fold AUC — TimeSeriesSplit")
axes[0].legend(loc="lower right", fontsize=8)

summary = cv_df.groupby("feature_set")["AUC"].agg(["mean", "std"]).reset_index()
axes[1].bar(summary["feature_set"], summary["mean"], yerr=summary["std"],
            color=["#4C72B0", "#DD8452"], capsize=6, width=0.5)
axes[1].set_ylim(0.68, 0.78)
axes[1].set_title("Mean ± std across 5 time folds")
axes[1].set_ylabel("AUC")
for i, row in summary.iterrows():
    axes[1].text(i, row["mean"] + 0.002, f'{row["mean"]:.4f}', ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print(summary.round(4).to_string(index=False))


**Takeaway:**  
| set | mean AUC | std |
|-----|----------|-----|
| all features | ≈ 0.7228 | ≈ 0.023 |
| dropped temporal | ≈ 0.7253 | ≈ 0.023 |

Difference **+0.0025** is well inside the fold noise (±0.023). Fold-wise results are mixed.  

**Decision: keep ALL features.** The single-split “win” from dropping was an artifact.  
(This also explains why an earlier submission that dropped features scored worse on the public leaderboard.)


## 9. Optuna hyperparameter search (LightGBM)

**Hypothesis:** a short search over learning rate, leaves, min child samples, subsample, colsample, and L2 can squeeze a little more AUC.

Search is done on the single 80/20 split for speed; the winner is then re-checked with TimeSeriesSplit.


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    p = {
        "objective": "binary",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7, 47),
        "min_child_samples": trial.suggest_int("min_child_samples", 30, 200),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0, log=True),
        "n_estimators": 2000,
        "random_state": 42,
        "verbose": -1,
    }
    m = lgb.LGBMClassifier(**p)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(80, verbose=False)],
    )
    return auc(y_va, m.predict_proba(X_va)[:, 1])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("best single-split AUC:", round(study.best_value, 4))
print("best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

best_params = {
    **study.best_params,
    "objective": "binary",
    "n_estimators": 2000,
    "random_state": 42,
    "verbose": -1,
}


In [ ]:
# Optuna history + param importances
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

trials_df = study.trials_dataframe()
axes[0].plot(trials_df["number"], trials_df["value"], "o-", color="#4C72B0", ms=4)
axes[0].axhline(base_auc, ls="--", color="gray", label=f"baseline {base_auc:.4f}")
axes[0].set_xlabel("trial")
axes[0].set_ylabel("val AUC")
axes[0].set_title("Optuna optimization history")
axes[0].legend()

try:
    importances = optuna.importance.get_param_importances(study)
    imp_s = pd.Series(importances).sort_values()
    imp_s.plot(kind="barh", ax=axes[1], color="#55A868")
    axes[1].set_title("Hyperparameter importance")
except Exception as e:
    axes[1].text(0.5, 0.5, f"importance n/a\n{e}", ha="center", va="center")
    axes[1].set_axis_off()
plt.tight_layout()
plt.show()

print("\n=== TimeSeriesSplit with Optuna params ===")
scores_opt = time_block_scores(all_cols)  # uses params via factory below — redefine factory
# proper check:
print("(re-run with best_params)")
def opt_factory():
    return lgb.LGBMClassifier(**best_params)

tscv = TimeSeriesSplit(n_splits=5)
scores_opt = []
for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), 1):
    m = opt_factory()
    m.fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=[(X.iloc[va_idx], y[va_idx])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    s = auc(y[va_idx], m.predict_proba(X.iloc[va_idx])[:, 1])
    scores_opt.append(s)
    print(f"  fold {fold}: AUC={s:.4f}")
print(f"  mean={np.mean(scores_opt):.4f} ± {np.std(scores_opt):.4f}")


**Takeaway:** Optuna best on one split ≈ **0.7548**.  
On TimeSeriesSplit: mean ≈ **0.7235** vs baseline **0.7228** — difference is noise.  

**Decision:** keep Optuna params for the final LightGBM (not worse than default, slightly more flexible).


## 10. Side experiment: model complexity

**Hypothesis:** simpler trees (fewer leaves / larger min_child) might generalize better under shift.

We compared three configs on the single 80/20 split early in the project.


In [ ]:
configs = {
    "baseline (leaves=31)": dict(num_leaves=31, min_child_samples=50,  colsample_bytree=0.7),
    "simpler (leaves=15)":  dict(num_leaves=15, min_child_samples=100, colsample_bytree=0.5),
    "simplest (leaves=7)":  dict(num_leaves=7,  min_child_samples=200, colsample_bytree=0.5),
}

comp_rows = []
for name, cfg in configs.items():
    p = {**params, **cfg, "n_estimators": 2000}
    m = lgb.LGBMClassifier(**p)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    s = auc(y_va, m.predict_proba(X_va)[:, 1])
    comp_rows.append({"config": name, "AUC": s, "trees": m.best_iteration_})
    print(f"{name:28s} AUC={s:.4f}  trees={m.best_iteration_}")

comp_df = pd.DataFrame(comp_rows)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(comp_df["config"], comp_df["AUC"], color=["#4C72B0", "#55A868", "#DD8452"])
ax.set_xlim(0.748, 0.754)
ax.set_xlabel("validation AUC")
ax.set_title("Complexity sweep (single time split)")
plt.tight_layout()
plt.show()
comp_df


**Takeaway:** `num_leaves=15` was slightly best among the three; we kept that family of settings (and later Optuna’s similar region).


## 11. Final model and submission

**Chosen recipe (best public score among 4 uploads → 1st place):**

1. **All features** (no temporal drop — rejected by TimeSeriesSplit).  
2. **LightGBM** with Optuna hyperparameters, averaged over **5 seeds**.  
3. **CatBoost** (depth 6, lr 0.03), averaged over **3 seeds**.  
4. **Rank blend** with LGBM weight **0.3** (robust region from section 5).  
5. Fit on the **entire** training set.

No stacking / third model in the final deliverable — two models were enough and more stable.


In [ ]:
W = 0.3  # LightGBM weight in rank blend

print("Training LightGBM seeds...")
p_lgb_test = np.zeros(len(X_test))
for seed in range(5):
    m = lgb.LGBMClassifier(**{**best_params, "random_state": seed})
    m.fit(X, y)
    p_lgb_test += m.predict_proba(X_test)[:, 1] / 5
    print(f"  LGBM seed {seed} done")

print("Training CatBoost seeds...")
p_cb_test = np.zeros(len(X_test))
for seed in range(3):
    c = CatBoostClassifier(
        iterations=800, learning_rate=0.03, depth=6,
        verbose=0, random_seed=seed,
    )
    c.fit(X, y)
    p_cb_test += c.predict_proba(X_test)[:, 1] / 3
    print(f"  CatBoost seed {seed} done")

pred_test = (W * rankdata(p_lgb_test) + (1 - W) * rankdata(p_cb_test)) / len(X_test)

# Sanity plots on test predictions
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(p_lgb_test, bins=40, color="#4C72B0", alpha=0.8)
axes[0].set_title("LGBM test scores")
axes[1].hist(p_cb_test, bins=40, color="#55A868", alpha=0.8)
axes[1].set_title("CatBoost test scores")
axes[2].hist(pred_test, bins=40, color="#C44E52", alpha=0.8)
axes[2].set_title("Rank-blend test scores")
for ax in axes:
    ax.set_xlabel("score")
plt.tight_layout()
plt.show()

sample = pd.read_csv(find("sample_submission.csv"))
sub = pd.DataFrame({sample.columns[0]: test["id"], sample.columns[1]: pred_test})
sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv", sub.shape)
sub.head()


## 12. Decision log (summary table)

| # | Experiment | Result | Decision |
|---|------------|--------|----------|
| 1 | Feature prep + PCA | usable matrix, ~91%/80% variance kept | keep |
| 2 | LGBM single time split | AUC ≈ 0.7518 | baseline |
| 3 | CatBoost + rank blend | AUC ≈ 0.7545 at w=0.3 | use blend |
| 4 | Adversarial validation | AUC ≈ 0.998, top = month & kw mins | investigate drop |
| 5 | Drop temporal features (1 split) | local AUC ↑ to ≈ 0.7549 | *candidate* |
| 6 | Same drop under TimeSeriesSplit | mean +0.0025 inside ±0.023 noise | **reject drop** |
| 7 | Optuna LGBM | +noise on time CV | keep params |
| 8 | Complexity sweep | leaves=15 competitive | aligned with Optuna |
| 9 | Final seed-averaged blend | best public score | **submit** |

### Why not drop time features?
Adversarial validation shows they *distinguish* train from test, but TimeSeriesSplit shows they still help predict the **target**. Removing them overfit the single validation block and hurt the leaderboard.

### Why rank blend?
AUC cares about ordering. Ranking each model’s scores before mixing is more robust than averaging calibrated probabilities from different libraries.

### Why multiple seeds?
Small datasets + bagging variance → averaging 3–5 seeds smooths the public score without changing the method.
